# SynBioCrow 2.2.0 RC2 — finalization build
This notebook builds RC2 from an **exact frozen source commit**. It is intentionally separate from the source commit so the generated M9 package and the eventual `v2.2.0-rc2` tag can point to the same commit.

RC2 change scope: stale registry test contract fixed; package metadata bumped to `2.2.0rc2`; no intentional scientific algorithm change from RC1.


In [ ]:
import os,sys,subprocess,shutil,json,hashlib,zipfile
from pathlib import Path
from google.colab import drive

REPO_URL="https://github.com/theiman112860/SynBioCrow.git"
SOURCE_COMMIT="631bac03b9a930a29721b955d1a06515942df23c"
ROOT=Path("/content/SynBioCrow_RC2")
if ROOT.exists(): shutil.rmtree(ROOT)

def run(cmd,cwd=None,timeout=3600):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if p.returncode:
        raise RuntimeError(f"command failed rc={p.returncode}: {cmd}")
    return p.stdout

run(["git","clone",REPO_URL,str(ROOT)])
run(["git","checkout",SOURCE_COMMIT],cwd=ROOT)
head=run(["git","rev-parse","HEAD"],cwd=ROOT).strip()
assert head==SOURCE_COMMIT,(head,SOURCE_COMMIT)
print("RC2 SOURCE COMMIT LOCK: PASS",head)


In [ ]:
# Install the exact RC2 source commit into the Colab runtime.
run([sys.executable,"-m","pip","install","-q","-e",str(ROOT)])
import importlib
importlib.invalidate_caches()
import synbiocrow
print("synbiocrow",synbiocrow.__file__)
run([sys.executable,"-m","pip","show","synbiocrow"])


In [ ]:
# Run the complete packaged unit-test suite. RC2 specifically must close the stale registry-contract failure from RC1.
run([sys.executable,"-m","pip","install","-q","pytest"])
run([sys.executable,"-m","pytest","-q","tests"],cwd=ROOT,timeout=1800)
print("RC2 TEST SUITE: PASS")


In [ ]:
# Verify release metadata and the standalone RetroPath certification certificate.
pyproject=(ROOT/"pyproject.toml").read_text()
assert 'version = "2.2.0rc2"' in pyproject
notes=(ROOT/"RELEASE_NOTES_2_2_RC2.md").read_text()
assert "stale unit-test assertion" in notes
cert=(ROOT/"docs/RETROPATH_STANDALONE_EQUIVALENCE.md").read_text()
for token in ["EXACT_COMPOUND_TRANSITION_MULTISET","historical rows: 34","standalone rows: 34","precision: 1.0","recall: 1.0","Jaccard: 1.0"]:
    assert token.lower() in cert.lower(),token
print("RC2 METADATA + RETROPATH CERTIFICATE: PASS")


In [ ]:
# Re-run M7 and M9 from the exact source commit.
run([sys.executable,str(ROOT/"scripts/m7_release_readiness.py")],cwd=ROOT,timeout=1800)
run([sys.executable,str(ROOT/"scripts/build_m9_release.py"),"--check"],cwd=ROOT,timeout=1800)
print("RC2 M7 + M9 CHECK: PASS")


In [ ]:
# Build a fresh RC2 reproducibility package directly to Google Drive.
drive.mount("/content/drive",force_remount=False)
OUT=Path("/content/drive/MyDrive/SynBioCrow/release/2.2.0-rc2")
OUT.mkdir(parents=True,exist_ok=True)
run([sys.executable,str(ROOT/"scripts/build_m9_release.py"),"--output-dir",str(OUT)],cwd=ROOT,timeout=1800)

def sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
    return h.hexdigest()

print("\nRC2 OUTPUTS")
for p in sorted(OUT.glob("*")):
    print(p.name,p.stat().st_size,sha256(p))

summary=json.loads((OUT/"SynBioCrow_2.2.0-rc2_PACKAGE_SUMMARY.json").read_text())
manifest=json.loads((OUT/"SynBioCrow_2.2.0-rc2_release_manifest.json").read_text())
assert summary["git_commit"]==SOURCE_COMMIT,summary["git_commit"]
assert manifest["git_commit"]==SOURCE_COMMIT,manifest["git_commit"]
print("RC2 PACKAGE COMMIT MATCH: PASS",SOURCE_COMMIT)
print("RC2 FINALIZATION: PASS")
print("Tag this exact commit as v2.2.0-rc2 after publishing the three generated M9 assets.")
